# Phase 2: NHANES Variable Recoding

**Purpose:** Convert NHANES "Refused" and "Don't know" codes to NaN across all variables. Each NHANES module uses different sentinel values depending on field width (single-digit 7/9, double-digit 77/99, triple-digit 777/999, etc.).

**Inputs:** `../data/processed/01_combined_nhanes.parquet`

**Output:** `../data/processed/02_recoded.parquet`

**Methodology:** Faithful port of the original recoding logic from the AIM-AHEAD ScHARe analysis. Variable lists are organized by NHANES module (Demographics, Medical Conditions, Spirometry, Food Security, Health Insurance, Health Status, Smoking, Early Childhood, Respiratory Health, Hospital Utilization, Physical Functioning) and grouped by their refused/don't-know coding pattern.

In [1]:
"""
Load the combined dataset from Phase 1.
The original notebook used `recode_df = combined_nhanes_common_only.copy()`;
we replicate that by reading the Phase 1 output and assigning to `recode_df`.
"""

from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("../data/processed")

recode_df = pd.read_parquet(PROCESSED_DIR / "01_combined_nhanes.parquet")

print(f"Loaded shape: {recode_df.shape[0]:,} rows × {recode_df.shape[1]} columns")
print(f"Columns sample: {list(recode_df.columns[:8])}")

Loaded shape: 30,442 rows × 474 columns
Columns sample: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIDEXMON', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDAGEEX']


In [2]:
import pandas as pd
import numpy as np

# =====================================
# DEMOGRAPHICS VARIABLES (DEMO_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/DEMO_E.htm
# =====================================

# Demographics: Single-digit codes (7=Refused, 9=Don't know)
demo_single_digit = [
    'DMQMILIT',  # Military status
    'DMDBORN2',  # Country of birth
    'DMDCITZN',  # Citizenship status
    'DMDEDUC2',  # Education level (adults 20+)
    'DMDSCHOL',  # School attendance
    'DMDHRBR2',  # HH reference person country of birth
    'DMDHREDU',  # HH reference person education
    'DMDHSEDU'   # HH reference person spouse education
]

# Demographics: Double-digit codes (77=Refused, 99=Don't know)
demo_double_digit = [
    'DMDYRSUS',  # Years in US
    'DMDEDUC3',  # Education level (youth 6-19)
    'DMDMARTL',  # Marital status
    'INDFMIN2',  # Family income
    'INDHHIN2',  # Household income
    'DMDHRMAR'   # HH reference person marital status
]

# =====================================
# MEDICAL CONDITIONS VARIABLES (MCQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/MCQ_E.htm
# =====================================

# Medical Conditions: Single-digit codes (7=Refused, 9=Don't know)
mcq_single_digit = [
    'MCQ010',    # Ever been told you have asthma
    'MCQ035',    # Still have asthma
    'MCQ040',    # Had asthma attack in past year
    'MCQ050',    # Emergency care visit for asthma
    'MCQ051',    # Taking treatment for anemia
    'MCQ053',    # Ever told you have anemia
    'MCQ080',    # Doctor ever said you were overweight
    'MCQ092',    # Ever receive blood transfusion
    'MCQ140',    # Trouble seeing even with glasses
    'MCQ149',    # Menstrual periods started yet
    'MCQ150G',   # School days missed
    'MCQ150Q',   # Doctor prescribe medication
    'MCQ160A',   # Ever told you have arthritis
    'MCQ160N',   # Ever told you had gout
    'MCQ160B',   # Ever told you had CHF
    'MCQ160C',   # Ever told you had CHD
    'MCQ160D',   # Ever told you had angina
    'MCQ160E',   # Ever told you had heart attack
    'MCQ160F',   # Ever told you had stroke
    'MCQ160G',   # Ever told you had emphysema
    'MCQ160M',   # Ever told you had thyroid problem
    'MCQ160K',   # Ever told you had chronic bronchitis
    'MCQ160L',   # Ever told you had liver condition
    'MCQ180M',   # Currently have thyroid problem
    'MCQ180K',   # Still have chronic bronchitis
    'MCQ180L',   # Still have liver condition
    'MCQ220',    # Ever told you had cancer
    'MCQ250A',   # Blood relatives have asthma
    'MCQ250B',   # Blood relatives have diabetes
    'MCQ260A',   # Blood relatives have heart attack
    'MCQ265',    # Blood relatives have osteoporosis
    'MCQ270',    # Blood relatives have hypertension
    'MCQ280',    # Blood relatives have heart attack
    'MCQ290',    # Parents have diabetes
    'MCQ300A',   # Close relative had heart attack - father
    'MCQ300B',   # Mother's father
    'MCQ300C',   # Father's father
    'MCQ300D',   # Brother
    'MCQ310',    # Ever have stroke/heart attack
    'MCQ320'     # Take preventive aspirin
]

# Medical Conditions: Double-digit codes (77=Refused, 99=Don't know)
mcq_double_digit = [
    'MCQ160AQ',  # Type of arthritis
    'MCQ240A',   # Type of cancer 1
    'MCQ240B',   # Type of cancer 2
    'MCQ240C',   # Type of cancer 3
    'MCQ240D',   # More than 3 kinds of cancer
    'MCQ240DK'   # First cancer type - Don't know
]

# Medical Conditions: Triple-digit codes (777=Refused, 999=Don't know)
mcq_triple_digit = [
    'MCQ320A',   # Age when difficulties began
    'MCQ330'     # How many days health was not good
]

# Medical Conditions: Five-digit codes (77777=Refused, 99999=Don't know)
mcq_five_digit = [
    'MCQ025',    # Age when first had asthma
    'MCQ150QQ',  # Days missed school
    'MCQ160NA',  # Age when told gout
    'MCQ160AA',  # Age when told arthritis
    'MCQ160BB',  # Age when told CHF
    'MCQ160CC',  # Age when told CHD
    'MCQ160DD',  # Age when told angina
    'MCQ160EE',  # Age when told heart attack
    'MCQ160FF',  # Age when told stroke
    'MCQ160GG',  # Age when told emphysema
    'MCQ160MM',  # Age when told thyroid problem
    'MCQ160KK',  # Age when told chronic bronchitis
    'MCQ160LL',  # Age when told liver condition
    'MCQ220LA',  # Age when told cancer
    # Age variables for different cancer types
    'MCQ240BA',  # Bladder cancer age
    'MCQ240BB',  # Blood cancer age
    'MCQ240BL',  # Bone cancer age
    'MCQ240BO',  # Brain cancer age
    'MCQ240BR',  # Breast cancer age
    'MCQ240CA',  # Cervical cancer age
    'MCQ240CO',  # Colon cancer age
    'MCQ240ES',  # Esophageal cancer age
    'MCQ240GA',  # Gallbladder cancer age
    'MCQ240KD',  # Kidney cancer age
    'MCQ240LA',  # Larynx cancer age
    'MCQ240LE',  # Leukemia age
    'MCQ240LI',  # Liver cancer age
    'MCQ240LU',  # Lung cancer age
    'MCQ240LY',  # Lymphoma age
    'MCQ240ME',  # Melanoma age
    'MCQ240MO',  # Mouth cancer age
    'MCQ240NE',  # Nervous system cancer age
    'MCQ240OV',  # Ovarian cancer age
    'MCQ240PA',  # Pancreatic cancer age
    'MCQ240PR',  # Prostate cancer age
    'MCQ240RE',  # Rectal cancer age
    'MCQ240SK',  # Skin cancer age
    'MCQ240SS',  # Soft tissue cancer age
    'MCQ240ST',  # Stomach cancer age
    'MCQ240TE',  # Testicular cancer age
    'MCQ240TH',  # Thyroid cancer age
    'MCQ240UT',  # Uterine cancer age
    'MCQ240YO'   # Other cancer age
]

# Medical Conditions: Six-digit codes (777777=Refused, 999999=Don't know)
mcq_six_digit = [
    'MCQ260AA'   # Days lost from work
]

# =====================================
# SPIROMETRY VARIABLES (SPX_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/SPX_E.htm
# =====================================

# Spirometry: Single-digit codes (7=Refused, 9=Don't know)
spx_single_digit = [
    'ENQ010',    # Breathing problem require oxygen?
    'ENQ020',    # Problem taking deep breath?
    'SPQ010',    # Have a current painful ear infection?
    'SPQ020',    # Have you/Has SP ever had eye surgery?
    'SPQ030',    # Eye surgery in the last 3 months?
    'SPQ040',    # Ever had open chest/abdominal surgery?
    'SPQ050',    # Chest/abdominal surgery last 3 months?
    'SPQ060',    # Tuberculosis in last year?
    'SPQ080',    # Stroke in the last 3 months?
    'SPQ090',    # Heart attack in last 3 months?
    'SPQ100',    # Coughed up blood past month?
    'ENQ100'     # Had respiratory illness?
]

# Spirometry: Double-digit codes (77=Refused, 99=Don't know)
spx_double_digit = [
    'SPQ070a'    # Ever told had an aneurysm?
]

# =====================================
# FOOD SECURITY VARIABLES (FSQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/FSQ_E.htm
# =====================================

# Food Security: Single-digit codes (7=Refused, 9=Don't know)
fsq_single_digit = [
    'FSD032A',   # HH Worried run out of food
    'FSD032B',   # HH Food didn't last
    'FSD032C',   # HH Couldn't afford balanced meals
    'FSD032D',   # HH Relied on low-cost food for child
    'FSD032E',   # HH Couldn't feed child balanced meal
    'FSD032F',   # HH Child not eating enough
    'FSD041',    # HH Adults cut size or skip meals
    'FSD052',    # HH How often adults cut size/skip meals
    'FSD061',    # HH Eat less than should
    'FSD071',    # HH Hungry, but didn't eat
    'FSD081',    # HH Lost weight, no money for food
    'FSD092',    # HH Adults not eat whole day
    'FSD102',    # HH How often adults not eat for day
    'FSD111',    # HH Cut size of child meals
    'FSD122',    # HH Child skip meals
    'FSD132',    # HH How often child skip meals
    'FSD141',    # HH Child hungry in last 12 months
    'FSD146',    # HH Child not eat whole day
    'FSD151',    # HH Emergency food received
    'FSD401',    # Meal size cut
    'FSD411',    # Skipped meals
    'FSD421',    # Ate less than should
    'FSD431',    # Hungry
    'FSQ440',    # Lost weight
    'FSD451',    # Not eat whole day
    'FSQ165',    # HH FS benefit: ever received
    'FSQ171',    # HH FS benefit: receive in last 12 months
    'FSQ162',    # HH WIC benefit: receive in last 12 month
    'FSD650ZC',  # CH WIC benefit: receive in last 12 month
    'FSD660ZC',  # CH WIC benefit: currently receive
    'FSD675',    # CH WIC benefit: received in infancy
    'FSD680',    # CH WIC benefit: received b/w 1-4 yrs old
    'FSQ690',    # CH WIC benefit: Mom received while preg.
    'FSD650ZW',  # WM WIC benefit: receive in last 12 month
    'FSD660ZW'   # WM WIC benefit: currently receive
]

# Food Security: Double-digit codes (77=Refused, 99=Don't know)
fsq_double_digit = [
    'FSQ695'     # CH WIC benefit: starting month of preg.
]

# Food Security: Triple-digit codes (777=Refused, 999=Don't know)
fsq_triple_digit = [
    'FSD670ZC',  # CH WIC benefit: # of months received
    'FSD670ZW'   # WM WIC benefit: # of months received
]

# Food Security: Five-digit codes (77777=Refused, 99999=Don't know)
fsq_five_digit = [
    'FSD225',    # HH FS benefit: time since last received
    'FSQ235'     # HH FS benefit: amount received last time
]

# =====================================
# HEALTH INSURANCE VARIABLES (HIQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/HIQ_E.htm
# =====================================

# Health Insurance: Single-digit codes (7=Refused, 9=Don't know)
hiq_single_digit = [
    'HIQ011',    # Covered by health insurance
    'HIQ260',    # Have Medicare?
    'HIQ270',    # Do plans cover prescriptions?
    'HIQ210'     # Time when no insurance in past year?
]

# Health Insurance: Double-digit codes (77=Refused, 99=Don't know)
hiq_double_digit = [
    'HIQ031A'    # Covered by private insurance
]

# =====================================
# HEALTH STATUS VARIABLES (HSQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/HSQ_E.htm
# =====================================

# Health Status: Single-digit codes (7=Refused, 9=Don't know)
hsq_single_digit = [
    'HSD010',    # General health condition
    'HSQ500',    # SP have head cold or chest cold
    'HSQ510',    # SP have stomach or intestinal illness?
    'HSQ520',    # SP have flu, pneumonia, ear infection?
    'HSQ571',    # SP donated blood in past 12 months?
    'HSQ590'     # Blood ever tested for HIV virus?
]

# Health Status: Double-digit codes (77=Refused, 99=Don't know)
hsq_double_digit = [
    'HSQ470',    # no. of days physical health was not good
    'HSQ480',    # no. of days mental health was not good
    'HSQ490',    # inactive days due to phys./mental hlth
    'HSQ493',    # Pain make it hard for usual activities
    'HSQ496',    # How many days feel anxious
    'HSQ580'     # How long ago was last blood donation?
]

# =====================================
# SMOKING VARIABLES (SMQ_E, SMQFAM_E, SMQRTU_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/SMQ_E.htm
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/SMQFAM_E.htm
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/SMQRTU_E.htm
# =====================================

# Smoking: Single-digit codes (7=Refused, 9=Don't know)
smq_single_digit = [
    # From SMQ_E
    'SMQ020',    # Smoked at least 100 cigarettes in life
    'SMQ040',    # Do you now smoke cigarettes
    'SMQ050U',   # Unit of measure (day/week/month/year)
    'SMQ077',    # How soon after waking do you smoke
    'SMQ620',    # Ever tried cigarette smoking
    'SMQ664M',   # Menthol or non-menthol Marlboro
    'SMQ664C',   # Menthol or non-menthol Camels
    'SMQ664W',   # Menthol or non-menthol Winston
    'SMQ664B',   # Menthol or non-menthol BensonHedges
    'SMQ664O',   # Menthol or non-menthol other brand
    'SMQ666M',   # Regular, light or ultralite Marlboro
    'SMQ666C',   # Regular, light or ultralite Camels
    'SMQ666K',   # Regular, light or ultralite Kools
    'SMQ666W',   # Regular, light or ultralite Winston
    'SMQ666B',   # Regular, light or ultralite BensonHedges
    'SMQ666S',   # Regular, light or ultralite Salem
    'SMQ666O',   # Regular, light or ultralite other brand
    'SMQ670',    # Tried to quit smoking
    # From SMQFAM_E
    'SMD410',    # Does anyone smoke inside home?
    # From SMQRTU_E
    'SMQ680',    # Used tobacco/nicotine last 5 days?
    'SMQ710',    # # days smoked cigarettes last 5 days
    'SMQ725',    # When did resp. smoke last cigarette?
    'SMQ740',    # # days smoked pipe over last 5 days
    'SMQ755',    # When did resp. smoke last pipe?
    'SMQ770',    # # days smoked cigars over last 5 days
    'SMQ785',    # When did resp. smoke last cigar?
    'SMQ800',    # #days used chewing tobacco -last 5 days
    'SMQ815',    # When did resp. last use chewing tobacco?
    'SMQ817',    # # days used snuff over last 5 days
    'SMQ819',    # When last used snuff
    'SMQ830',    # # days used nicotine stop smoking aids?
    'SMQ840'     # Last time used nicotine stop smoking aid
]

# Smoking: Double-digit codes (77=Refused, 99=Don't know)
smq_double_digit = [
    'SMD641',    # # days smoked cigs during past 30 days
    'SMD630',    # Age first smoked whole cigarette
    'SMQ660',    # Brands of cigarettes smoked/past mo
    'SMQ690A',   # Used last 5 days - Cigarettes
    'SMQ750',    # # pipes smoked per day - last 5 days
    'SMQ780'     # # cigars smoked per day - last 5 days
]

# Smoking: Triple-digit codes (777=Refused, 999=Don't know)
smq_triple_digit = [
    'SMD030',    # Age started smoking cigarettes regularly
    'SMD055',    # Age last smoked cigarettes regularly
    'SMD057',    # # cigarettes smoked per day when quit
    'SMD650',    # Avg # cigarettes/day during past 30 days
    'SMQ720'     # # cigarettes smoked per day
]

# Smoking: Five-digit codes (77777=Refused, 99999=Don't know)
smq_five_digit = [
    'SMQ050Q'    # How long since quit smoking cigarettes
]

# =====================================
# EARLY CHILDHOOD VARIABLES (ECQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/ECQ_E.htm
# =====================================

# Early Childhood: Single-digit codes (7=Refused, 9=Don't know)
ecq_single_digit = [
    'ECQ020',    # Mother smoked when pregnant
    'ECQ030',    # Mother quit smoking while pregnant
    'ECQ060',    # Receive newborn care at health facility
    'ECQ080',    # Weight more/less than 5.5 lbs
    'ECQ090',    # Weight more/less than 9.0 lbs
    'WHQ030E',   # How do you consider {SP} weight
    'MCQ080E',   # Doctor told you that {SP} was overweight
    'ECQ150',    # Doing anything to help control weight?
    'FSQ121'     # Now attend headstart
]

# Early Childhood: Double-digit codes (77=Refused, 99=Don't know)
ecq_double_digit = [
    'ECQ040'     # Mother quit smoking while pregnant (mo)
]

# Early Childhood: Four-digit codes (7777=Refused, 9999=Don't know)
ecq_four_digit = [
    'ECD010',    # Mother's age when born
    'ECD070A',   # Weight at birth, pounds
    'ECD070B'    # Weight at birth, ounces
]

# =====================================
# RESPIRATORY HEALTH VARIABLES (RDQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/RDQ_E.htm
# =====================================

# Respiratory Health: Single-digit codes (7=Refused, 9=Don't know)
rdq_single_digit = [
    'RDQ031',    # Coughing most days - over 3 mo period
    'RDQ050',    # Bring up phlegm most days - 3 mo period
    'RDQ070',    # Wheezing or whistling in chest - past yr
    'RDQ090',    # Wheezing disturb sleep in past year
    'RDQ100',    # Chest sound wheezy during exercise
    'RDQ134',    # Doctor prescribe wheezing medication
    'RDQ135',    # Amount wheezing limits usual activity
    'RDQ137',    # # Work/school days lost due to wheezing
    'RDQ140',    # Had dry cough at night in past year
    'AGQ030'     # Episode of hay fever in past 12 months?
]

# Respiratory Health: Four-digit codes (7777=Refused, 9999=Don't know)
rdq_four_digit = [
    'RDQ080',    # # wheezing/whistling attacks past year
    'RDD120'     # # medical visits for wheezing attacks
]

# Respiratory Health: Five-digit codes (77777=Refused, 99999=Don't know)
rdq_five_digit = [
    'RDD040',    # # years had cough problem
    'RDD060'     # # years bringing up phlegm problem
]

# =====================================
# HOSPITAL UTILIZATION VARIABLES (HUQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/HUQ_E.htm
# =====================================

# Hospital Utilization: Single-digit codes (7=Refused, 9=Don't know)
huq_single_digit = [
    'HUQ010',    # General health condition
    'HUQ020',    # Health now compared with 1 year ago
    'HUQ030',    # Routine place to go for healthcare
    'HUQ040',    # Type place most often go for healthcare
    'HUQ060',    # How long since last healthcare visit
    'HUQ071',    # Overnite hospital patient in last year
    'HUQ090'     # Seen mental health professional/past yr
]

# Hospital Utilization: Double-digit codes (77=Refused, 99=Don't know)
huq_double_digit = [
    'HUQ050'     # #times receive healthcare over past year
]

# Hospital Utilization: Five-digit codes (77777=Refused, 99999=Don't know)
huq_five_digit = [
    'HUD080'     # #times overnite hospital patient/last yr
]

# =====================================
# PHYSICAL FUNCTIONING VARIABLES (PFQ_E)
# From: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/PFQ_E.htm
# =====================================

# Physical Functioning: Single-digit codes (7=Refused, 9=Don't know)
pfq_single_digit = [
    'PFQ010',    # Physical,mental,emotional limitations
    'PFQ015',    # Able to take part in most type of play
    'PFQ020',    # Crawl, walk, run, play limitations
    'PFQ030',    # Long term impairment/health problem
    'PFQ041',    # Receive Special Ed or Early Intervention
    'PFQ049',    # Limitations keeping you from working
    'PFQ051',    # Limited in amount of work you can do
    'PFQ054',    # Need special equipment to walk
    'PFQ057',    # Experience confusion/memory problems
    'PFQ059',    # Physical, mental, emotional limitations
    'PFQ061A',   # Managing money difficulty
    'PFQ061B',   # Walking for a quarter mile difficulty
    'PFQ061C',   # Walking up ten steps difficulty
    'PFQ061D',   # Stooping, crouching, kneeling difficulty
    'PFQ061E',   # Lifting or carrying difficulty
    'PFQ061F',   # House chore difficulty
    'PFQ061G',   # Preparing meals difficulty
    'PFQ061H',   # Walking between rooms on same floor
    'PFQ061I',   # Standingup from armless chair difficulty
    'PFQ061J',   # Getting in and out of bed difficulty
    'PFQ061K',   # Using fork, knife, drinking from cup
    'PFQ061L',   # Dressing yourself difficulty
    'PFQ061M',   # Standing for long periods difficulty
    'PFQ061N',   # Sitting for long periods difficulty
    'PFQ061O',   # Reaching up over head difficulty
    'PFQ061P',   # Grasp/holding small objects difficulty
    'PFQ061Q',   # Going out to movies, events difficulty
    'PFQ061R',   # Attending social event difficulty
    'PFQ061S',   # Leisure activity at home difficulty
    'PFQ061T',   # Push or pull large objects difficulty
    'PFQ090'     # Require special healthcare equipment
]

# Physical Functioning: Double-digit codes (77=Refused, 99=Don't know)
pfq_double_digit = [
    'PFQ063A',   # Health problems causing difficulty
    'PFQ063B',   # Health problems causing difficulty
    'PFQ063C',   # Health problems causing difficulty
    'PFQ063D',   # Health problems causing difficulty
    'PFQ063E'    # Health problems causing difficulty
]

# Physical Functioning: Five-digit codes (77777=Refused, 99999=Don't know)
pfq_five_digit = [
    'PFD069A',   # Arthritis or rheumatism probl, days
    'PFD069B',   # Back or neck problems, days
    'PFD069C',   # Cancer condition, days
    'PFD069D',   # Depression/anxiety/emotional probl, days
    'PFD069E',   # Other development problems, days
    'PFD069F',   # Diabetes condition, days
    'PFD069G',   # Fractures/bone/joint injury probl, days
    'PFD069H',   # Hearing problems, days
    'PFD069I',   # Heart problems, days
    'PFD069J',   # Hypertension or high blood pressure,days
    'PFD069K',   # Lung or breathing problems, days
    'PFD069L',   # Mental retardation condition, days
    'PFD069M',   # Other injury problems, days
    'PFD069N',   # Senility condition, days
    'PFD069O',   # Stroke problems, days
    'PFD069P',   # Vision problems, days
    'PFD069Q',   # Weight problems, days
    'PFD069R'    # Other impairment problems, days
]

# Comprehensive recoding function with dataset tracking
def recode_nhanes_by_dataset(df):
    """
    Recode NHANES refused and don't know responses
    Track changes by dataset (Demographics, Medical Conditions, Spirometry, Food Security, 
    Health Insurance, Health Status, Smoking, Early Childhood, Respiratory Health, 
    Hospital Utilization, Physical Functioning)
    """
    df_recoded = df.copy()
    
    # Initialize summary tracking
    demo_summary = []
    mcq_summary = []
    spx_summary = []
    fsq_summary = []
    hiq_summary = []
    hsq_summary = []
    smq_summary = []
    ecq_summary = []
    rdq_summary = []
    huq_summary = []
    pfq_summary = []
    
    # Define all variable groups with their dataset source
    variable_groups = [
        # Demographics variables
        ('Demographics', demo_single_digit, [7, 9], 'Single (7/9)', demo_summary),
        ('Demographics', demo_double_digit, [77, 99], 'Double (77/99)', demo_summary),
        
        # Medical Conditions variables
        ('Medical Conditions', mcq_single_digit, [7, 9], 'Single (7/9)', mcq_summary),
        ('Medical Conditions', mcq_double_digit, [77, 99], 'Double (77/99)', mcq_summary),
        ('Medical Conditions', mcq_triple_digit, [777, 999], 'Triple (777/999)', mcq_summary),
        ('Medical Conditions', mcq_five_digit, [77777, 99999], 'Five-digit (77777/99999)', mcq_summary),
        ('Medical Conditions', mcq_six_digit, [777777, 999999], 'Six-digit (777777/999999)', mcq_summary),
        
        # Spirometry variables
        ('Spirometry', spx_single_digit, [7, 9], 'Single (7/9)', spx_summary),
        ('Spirometry', spx_double_digit, [77, 99], 'Double (77/99)', spx_summary),
        
        # Food Security variables
        ('Food Security', fsq_single_digit, [7, 9], 'Single (7/9)', fsq_summary),
        ('Food Security', fsq_double_digit, [77, 99], 'Double (77/99)', fsq_summary),
        ('Food Security', fsq_triple_digit, [777, 999], 'Triple (777/999)', fsq_summary),
        ('Food Security', fsq_five_digit, [77777, 99999], 'Five-digit (77777/99999)', fsq_summary),
        
        # Health Insurance variables
        ('Health Insurance', hiq_single_digit, [7, 9], 'Single (7/9)', hiq_summary),
        ('Health Insurance', hiq_double_digit, [77, 99], 'Double (77/99)', hiq_summary),
        
        # Health Status variables
        ('Health Status', hsq_single_digit, [7, 9], 'Single (7/9)', hsq_summary),
        ('Health Status', hsq_double_digit, [77, 99], 'Double (77/99)', hsq_summary),
        
        # Smoking variables
        ('Smoking', smq_single_digit, [7, 9], 'Single (7/9)', smq_summary),
        ('Smoking', smq_double_digit, [77, 99], 'Double (77/99)', smq_summary),
        ('Smoking', smq_triple_digit, [777, 999], 'Triple (777/999)', smq_summary),
        ('Smoking', smq_five_digit, [77777, 99999], 'Five-digit (77777/99999)', smq_summary),
        
        # Early Childhood variables
        ('Early Childhood', ecq_single_digit, [7, 9], 'Single (7/9)', ecq_summary),
        ('Early Childhood', ecq_double_digit, [77, 99], 'Double (77/99)', ecq_summary),
        ('Early Childhood', ecq_four_digit, [7777, 9999], 'Four-digit (7777/9999)', ecq_summary),
        
        # Respiratory Health variables
        ('Respiratory Health', rdq_single_digit, [7, 9], 'Single (7/9)', rdq_summary),
        ('Respiratory Health', rdq_four_digit, [7777, 9999], 'Four-digit (7777/9999)', rdq_summary),
        ('Respiratory Health', rdq_five_digit, [77777, 99999], 'Five-digit (77777/99999)', rdq_summary),
        
        # Hospital Utilization variables
        ('Hospital Utilization', huq_single_digit, [7, 9], 'Single (7/9)', huq_summary),
        ('Hospital Utilization', huq_double_digit, [77, 99], 'Double (77/99)', huq_summary),
        ('Hospital Utilization', huq_five_digit, [77777, 99999], 'Five-digit (77777/99999)', huq_summary),
        
        # Physical Functioning variables
        ('Physical Functioning', pfq_single_digit, [7, 9], 'Single (7/9)', pfq_summary),
        ('Physical Functioning', pfq_double_digit, [77, 99], 'Double (77/99)', pfq_summary),
        ('Physical Functioning', pfq_five_digit, [77777, 99999], 'Five-digit (77777/99999)', pfq_summary)
    ]
    
    print("RECODING NHANES REFUSED AND DON'T KNOW VALUES")
    print("=" * 80)
    
    # Process each variable group
    for dataset, vars_list, codes_to_replace, code_type, summary_list in variable_groups:
        if vars_list:  # Only process if there are variables
            print(f"\n{dataset} - {code_type} coded variables")
            print("-" * 60)
            
            variables_found = 0
            variables_recoded = 0
            total_values_recoded = 0
            
            for var in vars_list:
                if var in df_recoded.columns:
                    variables_found += 1
                    
                    # Count values before recoding
                    refused_count = (df_recoded[var] == codes_to_replace[0]).sum()
                    dont_know_count = (df_recoded[var] == codes_to_replace[1]).sum()
                    
                    if refused_count > 0 or dont_know_count > 0:
                        # Perform recoding
                        df_recoded[var] = df_recoded[var].replace(codes_to_replace, np.nan)
                        
                        # Report
                        print(f"  {var}: Refused={refused_count}, Don't know={dont_know_count}")
                        
                        variables_recoded += 1
                        total_values_recoded += refused_count + dont_know_count
                        
                        summary_list.append({
                            'Dataset': dataset,
                            'Variable': var,
                            'Coding': code_type,
                            'Refused': refused_count,
                            'Dont_Know': dont_know_count,
                            'Total_Recoded': refused_count + dont_know_count
                        })
            
            print(f"\n  Total variables in codebook: {len(vars_list)}")
            print(f"  Variables found in dataframe: {variables_found}")
            print(f"  Variables with recoded values: {variables_recoded}")
            print(f"  Total values recoded: {total_values_recoded}")
    
    return df_recoded, demo_summary, mcq_summary, spx_summary, fsq_summary, hiq_summary, hsq_summary, smq_summary, ecq_summary, rdq_summary, huq_summary, pfq_summary

# Apply the recoding to recode_df
recode_df, demo_summary, mcq_summary, spx_summary, fsq_summary, hiq_summary, hsq_summary, smq_summary, ecq_summary, rdq_summary, huq_summary, pfq_summary = recode_nhanes_by_dataset(recode_df)

# Create detailed summaries by dataset
print("\n" + "=" * 80)
print("SUMMARY BY DATASET")
print("=" * 80)

# Demographics summary
if demo_summary:
    demo_df = pd.DataFrame(demo_summary)
    print("\nDEMOGRAPHICS VARIABLES (DEMO_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(demo_df)}")
    print(f"Total refused values: {demo_df['Refused'].sum()}")
    print(f"Total don't know values: {demo_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {demo_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    demo_by_type = demo_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in demo_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\nDEMOGRAPHICS VARIABLES (DEMO_E)")
    print("No demographics variables had refused/don't know values to recode.")

# Medical Conditions summary
if mcq_summary:
    mcq_df = pd.DataFrame(mcq_summary)
    print("\n\nMEDICAL CONDITIONS VARIABLES (MCQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(mcq_df)}")
    print(f"Total refused values: {mcq_df['Refused'].sum()}")
    print(f"Total don't know values: {mcq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {mcq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    mcq_by_type = mcq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in mcq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nMEDICAL CONDITIONS VARIABLES (MCQ_E)")
    print("No medical conditions variables had refused/don't know values to recode.")

# Spirometry summary
if spx_summary:
    spx_df = pd.DataFrame(spx_summary)
    print("\n\nSPIROMETRY VARIABLES (SPX_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(spx_df)}")
    print(f"Total refused values: {spx_df['Refused'].sum()}")
    print(f"Total don't know values: {spx_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {spx_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    spx_by_type = spx_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in spx_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nSPIROMETRY VARIABLES (SPX_E)")
    print("No spirometry variables had refused/don't know values to recode.")

# Food Security summary
if fsq_summary:
    fsq_df = pd.DataFrame(fsq_summary)
    print("\n\nFOOD SECURITY VARIABLES (FSQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(fsq_df)}")
    print(f"Total refused values: {fsq_df['Refused'].sum()}")
    print(f"Total don't know values: {fsq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {fsq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    fsq_by_type = fsq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in fsq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nFOOD SECURITY VARIABLES (FSQ_E)")
    print("No food security variables had refused/don't know values to recode.")

# Health Insurance summary
if hiq_summary:
    hiq_df = pd.DataFrame(hiq_summary)
    print("\n\nHEALTH INSURANCE VARIABLES (HIQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(hiq_df)}")
    print(f"Total refused values: {hiq_df['Refused'].sum()}")
    print(f"Total don't know values: {hiq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {hiq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    hiq_by_type = hiq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in hiq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nHEALTH INSURANCE VARIABLES (HIQ_E)")
    print("No health insurance variables had refused/don't know values to recode.")

# Health Status summary
if hsq_summary:
    hsq_df = pd.DataFrame(hsq_summary)
    print("\n\nHEALTH STATUS VARIABLES (HSQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(hsq_df)}")
    print(f"Total refused values: {hsq_df['Refused'].sum()}")
    print(f"Total don't know values: {hsq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {hsq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    hsq_by_type = hsq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in hsq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nHEALTH STATUS VARIABLES (HSQ_E)")
    print("No health status variables had refused/don't know values to recode.")

# Smoking summary
if smq_summary:
    smq_df = pd.DataFrame(smq_summary)
    print("\n\nSMOKING VARIABLES (SMQ_E, SMQFAM_E, SMQRTU_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(smq_df)}")
    print(f"Total refused values: {smq_df['Refused'].sum()}")
    print(f"Total don't know values: {smq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {smq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    smq_by_type = smq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in smq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nSMOKING VARIABLES (SMQ_E, SMQFAM_E, SMQRTU_E)")
    print("No smoking variables had refused/don't know values to recode.")

# Early Childhood summary
if ecq_summary:
    ecq_df = pd.DataFrame(ecq_summary)
    print("\n\nEARLY CHILDHOOD VARIABLES (ECQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(ecq_df)}")
    print(f"Total refused values: {ecq_df['Refused'].sum()}")
    print(f"Total don't know values: {ecq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {ecq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    ecq_by_type = ecq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in ecq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nEARLY CHILDHOOD VARIABLES (ECQ_E)")
    print("No early childhood variables had refused/don't know values to recode.")

# Respiratory Health summary
if rdq_summary:
    rdq_df = pd.DataFrame(rdq_summary)
    print("\n\nRESPIRATORY HEALTH VARIABLES (RDQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(rdq_df)}")
    print(f"Total refused values: {rdq_df['Refused'].sum()}")
    print(f"Total don't know values: {rdq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {rdq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    rdq_by_type = rdq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in rdq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nRESPIRATORY HEALTH VARIABLES (RDQ_E)")
    print("No respiratory health variables had refused/don't know values to recode.")

# Hospital Utilization summary
if huq_summary:
    huq_df = pd.DataFrame(huq_summary)
    print("\n\nHOSPITAL UTILIZATION VARIABLES (HUQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(huq_df)}")
    print(f"Total refused values: {huq_df['Refused'].sum()}")
    print(f"Total don't know values: {huq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {huq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    huq_by_type = huq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in huq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nHOSPITAL UTILIZATION VARIABLES (HUQ_E)")
    print("No hospital utilization variables had refused/don't know values to recode.")

# Physical Functioning summary
if pfq_summary:
    pfq_df = pd.DataFrame(pfq_summary)
    print("\n\nPHYSICAL FUNCTIONING VARIABLES (PFQ_E)")
    print("-" * 60)
    print(f"Total variables with recoded values: {len(pfq_df)}")
    print(f"Total refused values: {pfq_df['Refused'].sum()}")
    print(f"Total don't know values: {pfq_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {pfq_df['Total_Recoded'].sum()}")
    
    # Summary by coding type
    pfq_by_type = pfq_df.groupby('Coding')['Total_Recoded'].sum()
    print("\nBy coding type:")
    for coding, total in pfq_by_type.items():
        print(f"  {coding}: {total} values")
else:
    print("\n\nPHYSICAL FUNCTIONING VARIABLES (PFQ_E)")
    print("No physical functioning variables had refused/don't know values to recode.")

# Combined summary
all_summary = demo_summary + mcq_summary + spx_summary + fsq_summary + hiq_summary + hsq_summary + smq_summary + ecq_summary + rdq_summary + huq_summary + pfq_summary
if all_summary:
    all_df = pd.DataFrame(all_summary)
    
    print("\n" + "=" * 80)
    print("GRAND TOTAL ACROSS ALL DATASETS")
    print("=" * 80)
    print(f"Total variables with recoded values: {len(all_df)}")
    print(f"Total refused values: {all_df['Refused'].sum()}")
    print(f"Total don't know values: {all_df['Dont_Know'].sum()}")
    print(f"TOTAL VALUES RECODED: {all_df['Total_Recoded'].sum()}")
    
    # Save detailed reports
    all_df.to_csv('nhanes_recoding_detailed.csv', index=False)
    
    # Save summary by dataset
    dataset_summary = all_df.groupby('Dataset').agg({
        'Variable': 'count',
        'Refused': 'sum',
        'Dont_Know': 'sum',
        'Total_Recoded': 'sum'
    }).rename(columns={'Variable': 'Variables_Count'})
    
    dataset_summary.to_csv('nhanes_recoding_by_dataset.csv')
    
    print("\nReports saved:")
    print("  - nhanes_recoding_detailed.csv (all variables with details)")
    print("  - nhanes_recoding_by_dataset.csv (summary by dataset)")

# Display total variable counts defined in code
print("\n" + "=" * 80)
print("TOTAL VARIABLES DEFINED IN CODE")
print("=" * 80)
print(f"\nDemographics (DEMO_E) variables:")
print(f"  Single-digit (7/9): {len(demo_single_digit)} variables")
print(f"  Double-digit (77/99): {len(demo_double_digit)} variables")
print(f"  TOTAL: {len(demo_single_digit) + len(demo_double_digit)} variables")

print(f"\nMedical Conditions (MCQ_E) variables:")
print(f"  Single-digit (7/9): {len(mcq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(mcq_double_digit)} variables")
print(f"  Triple-digit (777/999): {len(mcq_triple_digit)} variables")
print(f"  Five-digit (77777/99999): {len(mcq_five_digit)} variables")
print(f"  Six-digit (777777/999999): {len(mcq_six_digit)} variables")
print(f"  TOTAL: {len(mcq_single_digit) + len(mcq_double_digit) + len(mcq_triple_digit) + len(mcq_five_digit) + len(mcq_six_digit)} variables")

print(f"\nSpirometry (SPX_E) variables:")
print(f"  Single-digit (7/9): {len(spx_single_digit)} variables")
print(f"  Double-digit (77/99): {len(spx_double_digit)} variables")
print(f"  TOTAL: {len(spx_single_digit) + len(spx_double_digit)} variables")

print(f"\nFood Security (FSQ_E) variables:")
print(f"  Single-digit (7/9): {len(fsq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(fsq_double_digit)} variables")
print(f"  Triple-digit (777/999): {len(fsq_triple_digit)} variables")
print(f"  Five-digit (77777/99999): {len(fsq_five_digit)} variables")
print(f"  TOTAL: {len(fsq_single_digit) + len(fsq_double_digit) + len(fsq_triple_digit) + len(fsq_five_digit)} variables")

print(f"\nHealth Insurance (HIQ_E) variables:")
print(f"  Single-digit (7/9): {len(hiq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(hiq_double_digit)} variables")
print(f"  TOTAL: {len(hiq_single_digit) + len(hiq_double_digit)} variables")

print(f"\nHealth Status (HSQ_E) variables:")
print(f"  Single-digit (7/9): {len(hsq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(hsq_double_digit)} variables")
print(f"  TOTAL: {len(hsq_single_digit) + len(hsq_double_digit)} variables")

print(f"\nSmoking (SMQ_E, SMQFAM_E, SMQRTU_E) variables:")
print(f"  Single-digit (7/9): {len(smq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(smq_double_digit)} variables")
print(f"  Triple-digit (777/999): {len(smq_triple_digit)} variables")
print(f"  Five-digit (77777/99999): {len(smq_five_digit)} variables")
print(f"  TOTAL: {len(smq_single_digit) + len(smq_double_digit) + len(smq_triple_digit) + len(smq_five_digit)} variables")

print(f"\nEarly Childhood (ECQ_E) variables:")
print(f"  Single-digit (7/9): {len(ecq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(ecq_double_digit)} variables")
print(f"  Four-digit (7777/9999): {len(ecq_four_digit)} variables")
print(f"  TOTAL: {len(ecq_single_digit) + len(ecq_double_digit) + len(ecq_four_digit)} variables")

print(f"\nRespiratory Health (RDQ_E) variables:")
print(f"  Single-digit (7/9): {len(rdq_single_digit)} variables")
print(f"  Four-digit (7777/9999): {len(rdq_four_digit)} variables")
print(f"  Five-digit (77777/99999): {len(rdq_five_digit)} variables")
print(f"  TOTAL: {len(rdq_single_digit) + len(rdq_four_digit) + len(rdq_five_digit)} variables")

print(f"\nHospital Utilization (HUQ_E) variables:")
print(f"  Single-digit (7/9): {len(huq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(huq_double_digit)} variables")
print(f"  Five-digit (77777/99999): {len(huq_five_digit)} variables")
print(f"  TOTAL: {len(huq_single_digit) + len(huq_double_digit) + len(huq_five_digit)} variables")

print(f"\nPhysical Functioning (PFQ_E) variables:")
print(f"  Single-digit (7/9): {len(pfq_single_digit)} variables")
print(f"  Double-digit (77/99): {len(pfq_double_digit)} variables")
print(f"  Five-digit (77777/99999): {len(pfq_five_digit)} variables")
print(f"  TOTAL: {len(pfq_single_digit) + len(pfq_double_digit) + len(pfq_five_digit)} variables")

total_vars = (len(demo_single_digit) + len(demo_double_digit) + 
              len(mcq_single_digit) + len(mcq_double_digit) + len(mcq_triple_digit) + len(mcq_five_digit) + len(mcq_six_digit) +
              len(spx_single_digit) + len(spx_double_digit) +
              len(fsq_single_digit) + len(fsq_double_digit) + len(fsq_triple_digit) + len(fsq_five_digit) +
              len(hiq_single_digit) + len(hiq_double_digit) +
              len(hsq_single_digit) + len(hsq_double_digit) +
              len(smq_single_digit) + len(smq_double_digit) + len(smq_triple_digit) + len(smq_five_digit) +
              len(ecq_single_digit) + len(ecq_double_digit) + len(ecq_four_digit) +
              len(rdq_single_digit) + len(rdq_four_digit) + len(rdq_five_digit) +
              len(huq_single_digit) + len(huq_double_digit) + len(huq_five_digit) +
              len(pfq_single_digit) + len(pfq_double_digit) + len(pfq_five_digit))

print(f"\nGRAND TOTAL VARIABLES DEFINED: {total_vars} variables")

print("\n" + "=" * 80)
print("RECODING COMPLETE!")
print("=" * 80)






RECODING NHANES REFUSED AND DON'T KNOW VALUES

Demographics - Single (7/9) coded variables
------------------------------------------------------------
  DMQMILIT: Refused=1, Don't know=1
  DMDBORN2: Refused=11, Don't know=2
  DMDCITZN: Refused=47, Don't know=6
  DMDEDUC2: Refused=9, Don't know=18
  DMDSCHOL: Refused=0, Don't know=1
  DMDHRBR2: Refused=6, Don't know=9
  DMDHREDU: Refused=12, Don't know=97
  DMDHSEDU: Refused=11, Don't know=70

  Total variables in codebook: 8
  Variables found in dataframe: 8
  Variables with recoded values: 8
  Total values recoded: 301

Demographics - Double (77/99) coded variables
------------------------------------------------------------
  DMDYRSUS: Refused=134, Don't know=97
  DMDEDUC3: Refused=0, Don't know=1
  DMDMARTL: Refused=13, Don't know=2
  INDFMIN2: Refused=623, Don't know=541
  INDHHIN2: Refused=602, Don't know=558
  DMDHRMAR: Refused=161, Don't know=17

  Total variables in codebook: 6
  Variables found in dataframe: 6
  Variables wit

In [3]:
"""
Save the recoded dataset for use in Phase 3 (cleaning and filtering).
"""

output_path = PROCESSED_DIR / "02_recoded.parquet"
recode_df.to_parquet(output_path, index=False)

size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"Saved: {output_path}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {recode_df.shape}")

Saved: ..\data\processed\02_recoded.parquet
Size:  3.9 MB
Shape: (30442, 474)
